In [5]:
import sqlite3
import random
import string

SEED = 42
random.seed(SEED)

db_name = input("Enter SQLite DB filename: ").strip()
conn = sqlite3.connect(db_name)
cur = conn.cursor()

def rand_text(prefix="", length=8):
    return prefix + ''.join(random.choices(string.ascii_letters + string.digits, k=length))


def rand_bool():
    return random.choice([0, 1])

def run_sql(sql: str):
    try:
        cur = conn.execute(sql)
        if sql.lstrip().upper().startswith(("SELECT", "PRAGMA")):
            rows = cur.fetchall()
            return rows
        else:
            conn.commit()
            print("OK")
            return None

    except sqlite3.Error as err:
        print(f"SQLite error: {err}")
        return None


def main():
    
    try:
        cur.execute("PRAGMA foreign_keys = ON;")

        # --- Drop existing tables (idempotency) ---
        cur.executescript("""
        DROP TABLE IF EXISTS ISA;
        DROP TABLE IF EXISTS HAS;
        DROP TABLE IF EXISTS People;
        DROP TABLE IF EXISTS Office;
        DROP TABLE IF EXISTS Room;
        DROP TABLE IF EXISTS Building;
        """)

        # --- Create tables (interpreted from image) ---
        cur.executescript("""
        CREATE TABLE Building (
            name TEXT PRIMARY KEY,
            location TEXT,
            numRooms INT,
            accessibility BOOL,
            capacity INT
        );

        CREATE TABLE Room (
            number TEXT PRIMARY KEY,
            capacity INT,
            sqft INT,
            accessibility BOOL,
            building TEXT,
            FOREIGN KEY (building) REFERENCES Building(name)
        );

        CREATE TABLE Office (
            location TEXT PRIMARY KEY,
            department TEXT
        );

        CREATE TABLE People (
            id INT PRIMARY KEY,
            name TEXT,
            student BOOL,
            faculty BOOL,
            staff BOOL,
            department TEXT
        );

        CREATE TABLE HAS (
            id INT PRIMARY KEY,
            location TEXT,
            name TEXT,
            department TEXT,
            FOREIGN KEY (location) REFERENCES Office(location)
        );

        -- Interpreted: Office located in a Room (odd but allowed)
        CREATE TABLE ISA (
            location TEXT PRIMARY KEY,
            roomNumber TEXT,
            FOREIGN KEY (location) REFERENCES Office(location),
            FOREIGN KEY (roomNumber) REFERENCES Room(number)
        );
        """)

        # --- Insert Buildings ---
        buildings = []
        for i in range(100):
            name = f"B{i}_{rand_text('', 5)}"
            buildings.append(name)
            cur.execute("""
                INSERT INTO Building (name, location, numRooms, accessibility, capacity)
                VALUES (?, ?, ?, ?, ?)
            """, (
                name,
                random.choice([None, rand_text("Loc_", 6)]),  # nullable
                random.randint(-5, 500),  # negative rooms? schema allows it
                rand_bool(),
                random.randint(0, 10000)
            ))

        # --- Insert Rooms ---
        rooms = []
        for i in range(1000):
            number = f"R{i}_{rand_text('', 4)}"
            rooms.append(number)
            cur.execute("""
                INSERT INTO Room (number, capacity, sqft, accessibility, building)
                VALUES (?, ?, ?, ?, ?)
            """, (
                number,
                random.randint(0, 500),
                random.randint(-100, 10000),  # negative sqft—why not
                rand_bool(),
                random.choice(buildings)
            ))

        # --- Insert Offices ---
        offices = []
        for i in range(1000):
            loc = f"O{i}_{rand_text('', 4)}"
            offices.append(loc)
            cur.execute("""
                INSERT INTO Office (location, department)
                VALUES (?, ?)
            """, (
                loc,
                random.choice([
                    "CS", "Math", "History", "Physics",
                    "", None, rand_text("Dept_", 3)
                ])
            ))

        # --- Insert People ---
        for i in range(1000):
            cur.execute("""
                INSERT INTO People (id, name, student, faculty, staff, department)
                VALUES (?, ?, ?, ?, ?, ?)
            """, (
                i,
                random.choice([
                    rand_text("Name_", 5),
                    "",  # empty name allowed
                    None  # NULL name allowed
                ]),
                rand_bool(),
                rand_bool(),
                rand_bool(),
                random.choice(["CS", "Math", None, "", rand_text("Dept_", 2)])
            ))

        # --- Insert HAS ---
        for i in range(1000):
            cur.execute("""
                INSERT INTO HAS (id, location, name, department)
                VALUES (?, ?, ?, ?)
            """, (
                i,
                random.choice(offices),
                random.choice([
                    rand_text("Obj_", 5),
                    None,
                    ""
                ]),
                random.choice(["CS", None, "", rand_text("Dept_", 2)])
            ))

        # --- Insert ISA ---
        # One-to-one with Office due to PK on location
        for loc in offices:
            cur.execute("""
                INSERT INTO ISA (location, roomNumber)
                VALUES (?, ?)
            """, (
                loc,
                random.choice(rooms)
            ))

        conn.commit()
        print("Database populated successfully. Ship it.")

    except Exception as e:
        conn.rollback()
        print("Something went wrong, rolling back.")
        print(e)

    


if __name__ == "__main__":
    main()

Database populated successfully. Ship it.


In [9]:
run_sql("SELECT * FROM Building LIMIT 10;")

[('B0_cOtWM', None, 466, 0, 6451),
 ('B1_nnsik', None, 173, 0, 4862),
 ('B2_2PqO2', None, 368, 0, 4257),
 ('B3_wK6Iq', None, 426, 1, 1711),
 ('B4_sQxTd', None, 449, 1, 6656),
 ('B5_74g3o', None, 190, 1, 5333),
 ('B6_qz9wo', 'Loc_jvAhyF', 83, 0, 9568),
 ('B7_vEUKt', None, 233, 0, 3612),
 ('B8_fuE5c', 'Loc_rx6fZV', 39, 1, 5455),
 ('B9_cqVwA', 'Loc_rvOTi3', 373, 1, 6047)]